# Brinson模型绩效归因复现

本Notebook复现华泰证券研究所2020年8月发布的《Brinson模型基于持仓数据从类别配置、个券选择、交互作用进行绩效归因》研报核心内容。

## Brinson模型简介

Brinson模型由Brinson和Fachler于1985年提出，是基金绩效归因的经典模型。该模型将基金的超额收益分解为三个部分：

1. **类别配置收益 (Allocation Effect)**: 基金经理在行业/资产类别配置上的能力
2. **个券选择收益 (Selection Effect)**: 基金经理在个股选择上的能力  
3. **交互作用收益 (Interaction Effect)**: 类别配置与个券选择的联合作用

## 核心公式

### 四象限矩阵

|  | 实际组合资产类别j收益 | 基准组合资产类别j收益 |
|--|---------------------|---------------------|
| **实际组合资产类别j权重** | Q4 = ∑wp·rp | Q2 = ∑wp·rb |
| **基准组合资产类别j权重** | Q3 = ∑wb·rp | Q1 = ∑wb·rb |

### 收益分解

- **总超额收益**: R = Q4 - Q1
- **类别配置收益**: R_AA = Q2 - Q1 = ∑(wp - wb)·rb
- **个券选择收益**: R_SS = Q3 - Q1 = ∑(rp - rb)·wb
- **交互作用收益**: R_I = R - R_AA - R_SS = ∑(rp - rb)·(wp - wb)

## 1. 环境设置与导入

In [ ]:
# 导入必要的库
import sys
import os

# 添加source目录到路径
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'source'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from data_loader import FundDataLoader, DataProcessor
from factor import SinglePeriodBrinson, MultiPeriodBrinson, BrinsonAttributionAnalyzer
from backtest import BrinsonBacktest, BacktestConfig, BrinsonAnalysisReport
from plot import BrinsonVisualizer, create_full_report
from utils import print_attribution_summary, format_percentage

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ 环境设置完成")

## 2. 单期Brinson归因示例

首先展示单期Brinson模型的计算过程

In [ ]:
# 构造示例数据
sectors = ['金融', '科技', '消费', '医药', '能源']

# 实际组合权重和收益
portfolio_weights = pd.Series([0.25, 0.30, 0.20, 0.15, 0.10], index=sectors)
portfolio_returns = pd.Series([0.05, 0.10, 0.03, 0.08, -0.02], index=sectors)

# 基准组合权重和收益
benchmark_weights = pd.Series([0.20, 0.25, 0.25, 0.15, 0.15], index=sectors)
benchmark_returns = pd.Series([0.04, 0.08, 0.05, 0.06, -0.01], index=sectors)

print("实际组合权重:")
print(portfolio_weights)
print("\n基准组合权重:")
print(benchmark_weights)

In [ ]:
# 计算四象限
q1, q2, q3, q4 = SinglePeriodBrinson.calculate_four_quadrants(
    portfolio_weights, portfolio_returns,
    benchmark_weights, benchmark_returns
)

print("\n【Brinson四象限】")
print(f"Q1 (基准组合):       {q1*100:.2f}%")
print(f"Q2 (类别配置组合):   {q2*100:.2f}%")
print(f"Q3 (股票选择组合):   {q3*100:.2f}%")
print(f"Q4 (实际组合):       {q4*100:.2f}%")

In [ ]:
# 计算归因（三因素分解）
attr_3factor = SinglePeriodBrinson.calculate_attribution(
    portfolio_weights, portfolio_returns,
    benchmark_weights, benchmark_returns,
    include_interaction=True
)

print("\n【三因素分解结果】")
print_attribution_summary(attr_3factor.to_dict())

In [ ]:
# 计算归因（两因素分解，交互作用归入个券选择）
attr_2factor = SinglePeriodBrinson.calculate_attribution(
    portfolio_weights, portfolio_returns,
    benchmark_weights, benchmark_returns,
    include_interaction=False
)

print("\n【两因素分解结果】")
print_attribution_summary(attr_2factor.to_dict())

In [ ]:
# 各行业贡献明细
sector_contrib = SinglePeriodBrinson.calculate_sector_contribution(
    portfolio_weights, portfolio_returns,
    benchmark_weights, benchmark_returns
)

print("\n【各行业归因贡献】")
display(sector_contrib.round(4))

## 3. 可视化展示

In [ ]:
# 创建可视化器
visualizer = BrinsonVisualizer()

# 绘制归因瀑布图
fig1 = visualizer.plot_attribution_waterfall(
    attr_3factor.to_dict(),
    title="单期Brinson归因瀑布图"
)
plt.show()

In [ ]:
# 绘制行业贡献图
fig2 = visualizer.plot_sector_contribution(
    sector_contrib,
    title="各行业归因贡献"
)
plt.show()

In [ ]:
# 绘制归因摘要图
fig3 = visualizer.plot_attribution_summary(
    attr_3factor.to_dict(),
    title="归因结果摘要"
)
plt.show()

## 4. 多期Brinson归因回测

使用真实或模拟数据进行滚动归因分析

In [ ]:
# 设置回测参数
config = BacktestConfig(
    start_date='2023-01-01',
    end_date='2023-12-31',
    rebalance_freq='Q',  # 季度调仓
    include_interaction=True,
    benchmark_code='000300'  # 沪深300
)

# 创建回测引擎
backtest = BrinsonBacktest(config)

# 运行回测（使用模拟数据）
results_df = backtest.run_backtest('000001')

print("\n回测结果预览:")
display(results_df.round(4))

In [ ]:
# 计算多期累计归因
multi_period_attr = backtest.get_multi_period_attribution()

print("\n【多期累计归因结果】")
print(f"总超额收益:     {multi_period_attr.total*100:.2f}%")
print(f"类别配置收益:   {multi_period_attr.allocation*100:.2f}%")
print(f"个券选择收益:   {multi_period_attr.selection*100:.2f}%")
print(f"交互作用收益:   {multi_period_attr.interaction*100:.2f}%")

In [ ]:
# 计算绩效指标
metrics = backtest.calculate_performance_metrics()

print("\n【绩效统计】")
print(f"计算期数:           {metrics['total_periods']}")
print(f"配置收益胜率:       {metrics['allocation_win_rate']*100:.1f}%")
print(f"选择收益胜率:       {metrics['selection_win_rate']*100:.1f}%")
print(f"平均配置收益:       {metrics['avg_allocation']*100:.2f}%")
print(f"平均选择收益:       {metrics['avg_selection']*100:.2f}%")
print(f"总超额累计收益:     {metrics['total_cumulative_return']*100:.2f}%")
print(f"配置累计收益:       {metrics['allocation_cumulative']*100:.2f}%")
print(f"选择累计收益:       {metrics['selection_cumulative']*100:.2f}%")

In [ ]:
# 生成完整报告
report = BrinsonAnalysisReport(backtest)
print(report.generate_report())

## 5. 多期归因可视化

In [ ]:
# 绘制时间序列归因图
fig4 = visualizer.plot_time_series_attribution(results_df)
plt.show()

In [ ]:
# 绘制多期累计归因瀑布图
fig5 = visualizer.plot_attribution_waterfall(
    multi_period_attr.to_dict(),
    title="多期累计Brinson归因"
)
plt.show()

In [ ]:
# 获取行业贡献并绘制
sector_contributions = backtest.get_sector_contributions()

if not sector_contributions.empty:
    # 获取最新一期
    latest_date = sector_contributions['date'].max()
    latest_contrib = sector_contributions[sector_contributions['date'] == latest_date]
    
    fig6 = visualizer.plot_sector_contribution(
        latest_contrib,
        date=latest_date
    )
    plt.show()

## 6. 归因稳定性分析

In [ ]:
# 归因稳定性分析
stability = backtest.analyze_attribution_stability()

print("归因稳定性分析:")
display(stability.round(4))

In [ ]:
# 绘制稳定性图表
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 滚动均值
ax1 = axes[0]
ax1.plot(stability['date'], stability['allocation_mean']*100, label='配置收益', marker='o')
ax1.plot(stability['date'], stability['selection_mean']*100, label='选择收益', marker='s')
ax1.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax1.set_ylabel('滚动均值 (%)')
ax1.set_title('归因收益滚动均值')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 滚动标准差
ax2 = axes[1]
ax2.plot(stability['date'], stability['allocation_std']*100, label='配置收益', marker='o')
ax2.plot(stability['date'], stability['selection_std']*100, label='选择收益', marker='s')
ax2.set_ylabel('滚动标准差 (%)')
ax2.set_xlabel('日期')
ax2.set_title('归因收益滚动标准差')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 导出结果

In [ ]:
# 创建输出目录
import os
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

# 导出归因结果
results_df.to_csv(os.path.join(output_dir, 'attribution_results.csv'), index=False)
print(f"归因结果已导出至: {output_dir}/attribution_results.csv")

# 导出行业贡献
if not sector_contributions.empty:
    sector_contributions.to_csv(os.path.join(output_dir, 'sector_contributions.csv'), index=False)
    print(f"行业贡献已导出至: {output_dir}/sector_contributions.csv")

# 导出报告
report.save_report(os.path.join(output_dir, 'attribution_report.txt'))
print(f"文字报告已导出至: {output_dir}/attribution_report.txt")

In [ ]:
# 生成完整图表报告
create_full_report(
    results_df,
    multi_period_attr.to_dict(),
    sector_contributions if not sector_contributions.empty else pd.DataFrame(),
    output_dir=output_dir
)

## 8. 总结

本Notebook完整复现了Brinson模型的核心内容：

### 已实现功能

1. **单期Brinson归因**
   - 四象限矩阵计算
   - 三因素分解（配置、选择、交互）
   - 两因素分解（配置、选择）
   - 各行业贡献明细

2. **多期Brinson归因**
   - 几何链接法计算累计收益
   - 滚动归因分析
   - 绩效统计指标

3. **可视化展示**
   - 归因瀑布图
   - 行业贡献图
   - 时间序列图
   - 归因摘要图

4. **数据获取**
   - 支持efinance、akshare等数据源
   - 自动行业分类映射
   - 数据对齐与清洗

### 核心公式回顾

```
总超额收益:  R = Q4 - Q1
类别配置:    R_AA = (wp - wb) · rb
个券选择:    R_SS = (rp - rb) · wb
交互作用:    R_I = (rp - rb) · (wp - wb)
```

### 参考

- Brinson, G.P. and Fachler, N. (1985). Measuring non-US equity portfolio performance.
- 华泰证券研究所. (2020). Brinson模型基于持仓数据从类别配置、个券选择、交互作用进行绩效归因.